# Lesson 1.1 — From Business Problem to Claude Solution

**Level: Intermediate**

## Learning Objectives

By the end of this lesson, you will be able to:

- Translate an ambiguous business problem into a scoped Claude use case
- Map a candidate solution to the five business value pillars (efficiency, transformation, productivity, cost, performance/SLA)
- Write a one-page solution brief that a stakeholder can approve

**You will produce:** A `solution_brief.md` artifact that frames a business problem as a Claude solution with value-pillar scoring.

## Phase 1 Overview

Domain 1 (17% of the CCAR-P exam) is about architectural judgment: turning a messy business problem into an end-to-end Claude design and defending the pattern you chose. This phase is decision-heavy — you will produce briefs, architecture specs-as-code, pattern selection rubrics, and an Architecture Decision Record (ADR) you can reuse on the job and in the capstone.

By the end of this phase, you will have built:

- A stakeholder solution brief with value-pillar scoring
- An end-to-end architecture spec (input -> processing -> output -> feedback)
- A workflow-vs-agentic-vs-augmented-LLM selection rubric
- A multi-agent orchestration design
- A problem-decomposition worksheet and a final ADR

### Prerequisites

- Python 3 kernel; `pip install anthropic` (optional — cells mock without a key)
- 3+ years systems architecture experience (target audience)
- Read the CCAR-P exam guide, Domain 1 objectives

## Concepts

**Why start here?** Architects are paid for judgment, not code. The exam rewards candidates who can reason from a *business* problem to a *defensible* technical design.

**Key terms (defined on first use):**
- **LLM** — Large Language Model, e.g. Claude; a model that generates text from a prompt.
- **SLA** — Service Level Agreement; a measurable promise (e.g. p95 latency < 3s, 99.9% uptime).
- **Value pillars** — Anthropic frames business value as five levers: *efficiency* (do the same for less), *transformation* (do something previously impossible), *productivity* (amplify people), *cost* (unit economics), and *performance/SLA* (speed, reliability).

**The translation discipline:** business problem -> desired outcome -> measurable success metric -> candidate Claude capability -> risks. If you cannot state a success metric, you cannot design (or evaluate) the system. Scope narrowly; a crisp sub-problem beats a vague platform.

> **Reference:** [Anthropic — Building with Claude (overview)](https://docs.anthropic.com/en/docs/overview)

In [ ]:
# !pip install anthropic
# Guarded Claude client setup — every lesson runs with or without an API key.
import os

try:
    import anthropic
    if os.environ.get("ANTHROPIC_API_KEY"):
        client = anthropic.Anthropic()
        LIVE = True
    else:
        client = None
        LIVE = False
except Exception as e:  # SDK not installed in this kernel
    client = None
    LIVE = False
    print("anthropic SDK unavailable:", e)

# Current Claude model family (confirm exact IDs at https://docs.anthropic.com/en/docs/about-claude/models)
MODELS = {
    "opus": "claude-opus-4-20250514",    # deepest reasoning, highest cost/latency
    "sonnet": "claude-sonnet-4-20250514",# balanced default workhorse
    "haiku": "claude-3-5-haiku-20241022",# fastest/cheapest for simple tasks
}

def ask_claude(prompt, model=MODELS["sonnet"], system=None, max_tokens=512):
    """Live call when a key exists; otherwise a deterministic mock so notebooks run."""
    if not LIVE:
        return f"[MOCK {model}] would answer: {prompt[:80]}..."
    msg = client.messages.create(
        model=model, max_tokens=max_tokens,
        system=system or "You are a helpful solution architect.",
        messages=[{"role": "user", "content": prompt}],
    )
    return msg.content[0].text

print("LIVE" if LIVE else "MOCK", "mode ready")

## Code-Along: Step 1 — Capture the problem as structured intent

We model a business problem as data so the design stays testable. Fill the dataclass for a real-sounding case: a health insurer drowning in prior-authorization requests.

In [ ]:
from dataclasses import dataclass, field
from typing import List

@dataclass
class BusinessProblem:
    name: str
    stakeholder: str
    pain: str
    desired_outcome: str
    success_metric: str      # must be measurable
    constraints: List[str] = field(default_factory=list)

problem = BusinessProblem(
    name="Prior-authorization triage",
    stakeholder="VP Clinical Operations",
    pain="Nurses spend 15 min/request reading faxes and policy PDFs",
    desired_outcome="Draft an approve/deny/needs-info recommendation with citations",
    success_metric="Cut handle time 40% with >=95% agreement vs nurse on a gold set",
    constraints=["PHI/HIPAA", "human must approve final decision", "p95 < 20s"],
)
print(problem)

## Code-Along: Step 2 — Score the solution against the five value pillars

A one-number-per-pillar rubric forces explicit trade-offs and gives stakeholders a shared language. Scores are 0-5 with a short justification.

In [ ]:
pillars = {
    "efficiency":    (5, "40% less handle time directly removes labor cost"),
    "transformation":(2, "Not a new capability; augments an existing workflow"),
    "productivity":  (4, "Nurses handle more cases, focus on judgment calls"),
    "cost":          (3, "Token + review cost offset by labor savings; model later"),
    "performance":   (4, "p95<20s is achievable; SLA must be contractual"),
}
total = sum(s for s,_ in pillars.values())
for p,(s,why) in pillars.items():
    print(f"{p:14} {s}/5  {why}")
print(f"\nValue score: {total}/25 -> {'pursue' if total>=15 else 'reconsider'}")

## Code-Along: Step 3 — Draft and write the solution brief artifact

Turn intent + scoring into a one-page brief and persist it. This is the artifact a stakeholder signs off on before any engineering starts.

In [ ]:
def solution_brief(p, pillars):
    lines = [f"# Solution Brief: {p.name}", "",
             f"**Stakeholder:** {p.stakeholder}", "",
             f"## Problem\n{p.pain}", "",
             f"## Desired outcome\n{p.desired_outcome}", "",
             f"## Success metric (measurable)\n{p.success_metric}", "",
             "## Constraints"]
    lines += [f"- {c}" for c in p.constraints]
    lines += ["", "## Value-pillar scoring"]
    lines += [f"- **{k}**: {s}/5 — {w}" for k,(s,w) in pillars.items()]
    return "\n".join(lines)

brief = solution_brief(problem, pillars)
with open("solution_brief.md", "w", encoding="utf-8") as fh:
    fh.write(brief)
print(brief[:400])
print("\n...written to solution_brief.md")

## Exercise

Pick a different problem (e.g. *contract-clause extraction for a legal team* or *tier-1 IT support deflection*). Create a `BusinessProblem`, score all five pillars, and write a brief. **The hard part is the success metric — make it measurable.**

In [ ]:
# TODO: define your own BusinessProblem
my_problem = BusinessProblem(
    name="...",
    stakeholder="...",
    pain="...",
    desired_outcome="...",
    success_metric="...",  # TODO: must be measurable (number + gold reference)
    constraints=[],         # TODO
)

# TODO: score the five pillars as {pillar: (score, justification)}
my_pillars = {}

# TODO: write the brief to my_brief.md and print the value score

## Key Takeaways

- Architecture starts with a measurable success metric, not a model choice.
- The five value pillars give stakeholders a shared vocabulary for trade-offs.
- A one-page brief is the cheapest artifact to get wrong early and fix.
- Scope a sharp sub-problem; vague platforms fail standard-setting-level scrutiny.

---

**Next:** [Lesson 1.2 — Designing the End-to-End Architecture](./lesson-1.2-end-to-end-architecture.ipynb)